In [76]:
import numpy as np
from tqdm import tqdm
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import re
from gensim.models import Word2Vec
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [77]:
# Load and preprocess data
df = pd.read_csv('/kaggle/input/amazon-fine-food-reviews/Reviews.csv')

In [78]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)
df['Sentiment'] = df['Score'].apply(lambda x: 0 if x <= 2 else (1 if x >= 4 else 2))
df = df[df['Sentiment'] != 2]  # Remove neutral reviews

In [79]:
from gensim.utils import simple_preprocess
tokenized_text = [simple_preprocess(text) for text in df['Cleaned_Text']]

In [80]:
import gensim.downloader as api  # Import the correct module

print("Loading pre-trained Word2Vec model...")
word2vec_model = api.load('word2vec-google-news-300')  # Now this should work
embedding_dim = 300  # Google News vectors are 300-dimensional


Loading pre-trained Word2Vec model...


In [81]:
vectorizer = TfidfVectorizer(max_features=100)
X_tfidf = vectorizer.fit_transform(df['Cleaned_Text'])
vocab = vectorizer.get_feature_names_out()

In [82]:
def simple_bca(X, y, vocab, num_features=10, n_iter=3):
    """
    Simplified Binary Coordinate Ascent for feature selection with tqdm progress bars
    Args:
        X: TF-IDF matrix (sparse)
        y: Target labels
        vocab: Feature names
        num_features: Number of features to select
        n_iter: Number of iterations
    Returns:
        Indices of selected features
    """
    # Initialize with random features
    selected = set(np.random.choice(len(vocab), size=num_features, replace=False))
    best_score = 0
    
    # Use a simple model for evaluation
    model = LogisticRegression(max_iter=100, solver='liblinear')
    
    for _ in tqdm(range(n_iter), desc="BCA Iterations"):
        improved = False
        
        # Evaluate removing each feature with progress bar
        for feature in tqdm(list(selected), desc="Removing Features", leave=False):
            temp = selected - {feature}
            if not temp:
                continue
                
            X_temp = X[:, list(temp)].toarray()
            score = np.mean(cross_val_score(model, X_temp, y, cv=3, scoring='accuracy'))
            
            if score > best_score:
                best_score = score
                selected = temp
                improved = True
        
        # Evaluate adding top potentially helpful features with progress bar
        remaining = set(range(len(vocab))) - selected
        if remaining:
            # Score potential additions (using mutual information for speed)
            mi_scores = mutual_info_classif(X[:, list(remaining)], y, discrete_features=True)
            top_candidates = np.array(list(remaining))[np.argsort(mi_scores)[-5:]]  # Top 5
            
            for candidate in tqdm(top_candidates, desc="Adding Features", leave=False):
                temp = selected | {candidate}
                X_temp = X[:, list(temp)].toarray()
                score = np.mean(cross_val_score(model, X_temp, y, cv=3, scoring='accuracy'))
                
                if score > best_score:
                    best_score = score
                    selected = temp
                    improved = True
        
        if not improved:
            break
    
    return np.array(list(selected))

In [83]:
print("Running feature selection...")
selected_indices = simple_bca(X_tfidf, df['Sentiment'].values, vocab, num_features=5)
selected_words = vocab[selected_indices]

Running feature selection...


Removing Features: 100%|██████████| 5/5 [00:03<00:00,  1.30it/s]
                                                                /usr/local/lib/python3.10/dist-packages/sklearn/metrics/cluster/_supervised.py:64: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/cluster/_supervised.py:64: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/cluster/_supervised.py:64: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/cluster/_supervised.py:64: UserWarning: Clustering metrics expects discrete values 

In [84]:
# Filter tokenized text to only include selected words
filtered_tokenized = [[word for word in doc if word in selected_words] for doc in tokenized_text]

In [85]:
vocab_size = len(selected_words) + 1  # +1 for padding token
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for i, word in enumerate(selected_words):
    if word in word2vec_model:
        embedding_matrix[i+1] = word2vec_model[word]  # index 0 is for padding

In [86]:
from tensorflow.keras.preprocessing.text import Tokenizer


tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts([' '.join(words) for words in filtered_tokenized])
sequences = tokenizer.texts_to_sequences([' '.join(words) for words in filtered_tokenized])

In [87]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 200
X = pad_sequences(sequences, maxlen=max_len)
y = df['Sentiment'].values

In [88]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Hybrid CNN-RNN Model with Word2Vec embeddings
def build_hybrid_model(input_shape, embedding_matrix, vocab_size, embedding_dim):
    inputs = Input(shape=input_shape)
    
    # Embedding layer with Word2Vec weights
    embedding = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=input_shape[0],
        trainable=False # Set to True if you want to fine-tune
    )(inputs)
    
    # CNN Branch
    conv1 = Conv1D(64, kernel_size=3, activation='relu', padding='same')(embedding)
    pool1 = MaxPooling1D(pool_size=2)(conv1)
    
    conv2 = Conv1D(64, kernel_size=3, activation='relu', padding='same')(pool1)
    pool2 = MaxPooling1D(pool_size=2)(conv2)
    
    # RNN Branch
    lstm1 = LSTM(64, return_sequences=True)(embedding)
    lstm2 = LSTM(32)(lstm1)
    
    # Concatenate branches
    cnn_flatten = tf.keras.layers.Flatten()(pool2)
    merged = concatenate([cnn_flatten, lstm2])
    
    # Classifier
    dense1 = Dense(64, activation='relu')(merged)
    dropout1 = Dropout(0.5)(dense1)
    dense2 = Dense(32, activation='relu')(dropout1)
    dropout2 = Dropout(0.3)(dense2)
    outputs = Dense(1, activation='sigmoid')(dropout2)
    
    return Model(inputs=inputs, outputs=outputs)

In [89]:
model = build_hybrid_model(
    input_shape=(max_len,),
    embedding_matrix=embedding_matrix,
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [90]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Define learning rate reduction callback
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',  # Monitor validation loss
    factor=0.9,         # Reduce learning rate by half
    patience=3,         # Number of epochs with no improvement before reducing LR
    min_lr=1e-6,        # Minimum learning rate
    verbose=1           # Show messages when LR changes
)

# Compile model with original learning rate
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',  # Fixed typo in your original code ('binary_crossentropy')
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model.summary()

# Train the model with learning rate reduction
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_data=(X_test, y_test),
    class_weight={0: 3.0, 1: 1.0},
    callbacks=[reduce_lr]  # Add the learning rate reduction callback
)

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7             │ (None, 200)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_7 (Embedding)   │ (None, 200, 300)       │          1,500 │ input_layer_7[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_14 (Conv1D)        │ (None, 200, 64)        │         57,664 │ embedding_7[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling1d_14          │ (None, 100, 64)        │              0 │ conv1d_14[0][0]        │
│ (MaxPooling1D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_15 (Conv1D)        │ (None, 100, 64)        │         12,352 │ max_pooling1d_14[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling1d_15          │ (None, 50, 64)         │              0 │ conv1d_15[0][0]        │
│ (MaxPooling1D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_14 (LSTM)            │ (None, 200, 64)        │         93,440 │ embedding_7[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_7 (Flatten)       │ (None, 3200)           │              0 │ max_pooling1d_15[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_15 (LSTM)            │ (None, 32)             │         12,416 │ lstm_14[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_7             │ (None, 3232)           │              0 │ flatten_7[0][0],       │
│ (Concatenate)             │                        │                │ lstm_15[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_21 (Dense)          │ (None, 64)             │        206,912 │ concatenate_7[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_14 (Dropout)      │ (None, 64)             │              0 │ dense_21[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_22 (Dense)          │ (None, 32)             │          2,080 │ dropout_14[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_15 (Dropout)      │ (None, 32)             │              0 │ dense_22[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_23 (Dense)          │ (None, 1)              │             33 │ dropout_15[0][0]       │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 386,397 (1.47 MB)

 Trainable params: 384,897 (1.47 MB)

 Non-trainable params: 1,500 (5.86 KB)

Epoch 1/5
3287/3287 ━━━━━━━━━━━━━━━━━━━━ 105s 31ms/step - accuracy: 0.8422 - loss: 0.8585 - precision_7: 0.8438 - recall_7: 0.9977 - val_accuracy: 0.8446 - val_loss: 0.5343 - val_precision_7: 0.8446 - val_recall_7: 1.0000 - learning_rate: 0.0010
Epoch 2/5
3287/3287 ━━━━━━━━━━━━━━━━━━━━ 101s 31ms/step - accuracy: 0.8439 - loss: 0.8526 - precision_7: 0.8445 - recall_7: 0.9991 - val_accuracy: 0.8446 - val_loss: 0.5334 - val_precision_7: 0.8446 - val_recall_7: 1.0000 - learning_rate: 0.0010
Epoch 3/5
3287/3287 ━━━━━━━━━━━━━━━━━━━━ 100s 31ms/step - accuracy: 0.8432 - loss: 0.8532 - precision_7: 0.8440 - recall_7: 0.9988 - val_accuracy: 0.8446 - val_loss: 0.5348 - val_precision_7: 0.8446 - val_recall_7: 0.9998 - learning_rate: 0.0010
Epoch 4/5
3287/3287 ━━━━━━━━━━━━━━━━━━━━ 100s 31ms/step - accuracy: 0.8429 - loss: 0.8532 - precision_7: 0.8439 - recall_7: 0.9985 - val_accuracy: 0.8447 - val_loss: 0.5279 - val_precision_7: 0.8447 - val_recall_7: 0.9999 - learning_rate: 0.0010
Epoch 5/5
3287/3

In [91]:
from sklearn.metrics import accuracy_score

# Get predictions
y_pred = (model.predict(X_test) > 0.5).astype(int)

# Calculate and print test accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# # Print classification report
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred))

# # Print confusion matrix
# print("\nConfusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

3287/3287 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step
Test Accuracy: 0.8448


In [92]:
# Save models
word2vec_model.save("word2vec.model")
model.save("hybrid_cnn_rnn_word2vec.h5")

# Novelty

In [2]:
import numpy as np
import pandas as pd
import re
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
# Load and preprocess data
df = pd.read_csv('/kaggle/input/amazon-fine-food-reviews/Reviews.csv')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)
df['Sentiment'] = df['Score'].apply(lambda x: 0 if x <= 2 else (1 if x >= 4 else 2))
df = df[df['Sentiment'] != 2]  # Remove neutral reviews

In [4]:
# Tokenization
max_words = 20000  # Vocabulary size
max_len = 200      # Maximum sequence length

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(df['Cleaned_Text'])
sequences = tokenizer.texts_to_sequences(df['Cleaned_Text'])
X = pad_sequences(sequences, maxlen=max_len)
y = df['Sentiment'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Transformer components
class MultiHeadSelfAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads=8):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"embedding dimension = {embed_dim} should be divisible by number of heads = {num_heads}"
            )
        self.projection_dim = embed_dim // num_heads
        self.query_dense = Dense(embed_dim)
        self.key_dense = Dense(embed_dim)
        self.value_dense = Dense(embed_dim)
        self.combine_heads = Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)
        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim),]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=None):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = tf.keras.layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.pos_emb = tf.keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [8]:
# Build the Transformer model
embed_dim = 128  # Embedding size for each token
num_heads = 8    # Number of attention heads
ff_dim = 128     # Hidden layer size in feed forward network inside transformer

inputs = Input(shape=(max_len,))
embedding_layer = TokenAndPositionEmbedding(max_len, max_words, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(32, activation="relu")(x)
x = Dropout(0.1)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)

In [9]:
# Compile and train
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.9,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=5,
    validation_data=(X_test, y_test),
    callbacks=[reduce_lr],
    class_weight={0: 3.0, 1: 1.0}  # Adjust for class imbalance
)

Epoch 1/5
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 110s 8ms/step - accuracy: 0.8935 - loss: 0.4137 - precision: 0.9538 - recall: 0.9186 - val_accuracy: 0.9309 - val_loss: 0.1735 - val_precision: 0.9751 - val_recall: 0.9423 - learning_rate: 0.0010
Epoch 2/5
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 96s 7ms/step - accuracy: 0.9351 - loss: 0.2486 - precision: 0.9794 - recall: 0.9430 - val_accuracy: 0.9359 - val_loss: 0.1747 - val_precision: 0.9769 - val_recall: 0.9464 - learning_rate: 0.0010
Epoch 3/5
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 97s 7ms/step - accuracy: 0.9444 - loss: 0.2148 - precision: 0.9816 - recall: 0.9520 - val_accuracy: 0.9432 - val_loss: 0.1508 - val_precision: 0.9729 - val_recall: 0.9594 - learning_rate: 0.0010
Epoch 4/5
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 97s 7ms/step - accuracy: 0.9519 - loss: 0.1873 - precision: 0.9847 - recall: 0.9579 - val_accuracy: 0.9379 - val_loss: 0.1756 - val_precision: 0.9792 - val_recall: 0.9466 - learning_rate: 0.0010
Epoch 5/5
13146/13146 ━━━━━━━━━━━━━━━━━━━━ 96s 

In [10]:
# Evaluate
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


3287/3287 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
Test Accuracy: 0.9354

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.90      0.81     16379
           1       0.98      0.94      0.96     88784

    accuracy                           0.94    105163
   macro avg       0.86      0.92      0.89    105163
weighted avg       0.94      0.94      0.94    105163


Confusion Matrix:
[[14759  1620]
 [ 5175 83609]]


In [11]:

# Save model
model.save("transformer_sentiment_model.h5")